# Subgraphs in LangGraph

Subgraphs let you compose graphs inside other graphs. They are the primary mechanism for building **modular, multi-agent systems** in LangGraph — each agent (or team of agents) lives in its own subgraph, and a parent graph orchestrates how they collaborate.

In [ ]:
# %pip install langgraph

## What are Subgraphs?

A **subgraph** is just a compiled LangGraph that you embed inside another graph. There are two main reasons to use them:

1. **Composition** — build complex graphs out of smaller, self-contained pieces
2. **Multi-agent systems** — each agent is a subgraph; the parent graph routes between them

There are two patterns for embedding a subgraph, depending on whether it shares state keys with the parent or has its own private schema.

## Pattern 1 — Shared State Keys

When the subgraph's state schema **shares keys** with the parent graph, you can add the compiled subgraph **directly as a node**. LangGraph will automatically pass the shared keys in and merge the output keys back out.

This is the simplest and most common pattern.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# Subgraph state (shares 'messages' key with parent)
class SubgraphState(TypedDict):
    messages: list
    sub_result: str

def sub_node(state: SubgraphState):
    last = state["messages"][-1] if state["messages"] else "none"
    return {"sub_result": f"Processed: {last}"}

sub_builder = StateGraph(SubgraphState)
sub_builder.add_node("sub_node", sub_node)
sub_builder.add_edge(START, "sub_node")
sub_builder.add_edge("sub_node", END)
subgraph = sub_builder.compile()

# Parent graph
class ParentState(TypedDict):
    messages: list
    sub_result: str
    final: str

def finalize(state: ParentState):
    return {"final": f"Done: {state['sub_result']}"}

parent_builder = StateGraph(ParentState)
parent_builder.add_node("subgraph", subgraph)   # subgraph added directly as node
parent_builder.add_node("finalize", finalize)
parent_builder.add_edge(START, "subgraph")
parent_builder.add_edge("subgraph", "finalize")
parent_builder.add_edge("finalize", END)
parent = parent_builder.compile()

result = parent.invoke({"messages": ["hello"], "sub_result": "", "final": ""})
print(result)

## Pattern 2 — Different Schemas

When the subgraph has a **completely different state schema** from the parent, you cannot add it directly. Instead, wrap it inside a regular node function that:

1. Translates parent state → subgraph input
2. Invokes the subgraph
3. Translates subgraph output → parent state update

This pattern is useful when you want a subgraph to keep its internals **private** — only exposing a clean input/output contract to the parent.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# Subgraph with its own private state
class AnalysisState(TypedDict):
    text: str
    word_count: int

def count_words(state: AnalysisState):
    return {"word_count": len(state["text"].split())}

analysis_builder = StateGraph(AnalysisState)
analysis_builder.add_node("count_words", count_words)
analysis_builder.add_edge(START, "count_words")
analysis_builder.add_edge("count_words", END)
analysis_subgraph = analysis_builder.compile()

# Parent graph with different schema — invoke subgraph inside a node
class MainState(TypedDict):
    content: str
    analysis: dict

def run_analysis(state: MainState):
    # Translate parent state → subgraph state
    sub_result = analysis_subgraph.invoke({"text": state["content"], "word_count": 0})
    return {"analysis": sub_result}

main_builder = StateGraph(MainState)
main_builder.add_node("run_analysis", run_analysis)
main_builder.add_edge(START, "run_analysis")
main_builder.add_edge("run_analysis", END)
main_graph = main_builder.compile()

result = main_graph.invoke({"content": "Hello world this is LangGraph", "analysis": {}})
print(result)

## Persistence in Subgraphs

**Per-thread persistence is the default and recommended approach.** When you compile the parent graph with a checkpointer, every subgraph invocation within that thread is automatically checkpointed too — no extra wiring needed.

Per-invocation persistence (where state is reset on every call) is rarely useful for stateful multi-agent systems. Use per-thread persistence when you want subgraph agents to remember context across turns of a conversation.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# Compile the parent graph (from Pattern 1) with a checkpointer
memory = MemorySaver()
checkpointed_parent = parent_builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "agent-thread-1"}}
result = checkpointed_parent.invoke(
    {"messages": ["hello"], "sub_result": "", "final": ""},
    config=config,
)
print(result)

# Subgraph state is persisted as part of the parent thread
state = checkpointed_parent.get_state(config)
print("Persisted state:", state.values)

## When to Use Subgraphs

Reach for subgraphs when you need:

- **Complex multi-agent teams** — each agent or team is its own subgraph, the parent orchestrates
- **Reusable agent modules** — package a subgraph once, drop it into many parent graphs
- **Isolating failure domains** — a crash inside a subgraph node is easier to reason about than a crash inside a giant monolithic graph
- **Different teams owning different components** — each team ships and tests its subgraph independently

If your graph is small and the logic is linear, you probably don't need subgraphs yet. Reach for them when complexity demands modularity.